# Transfer Learning - Cats vs Dogs
## Projeto de Transfer Learning para Classificação de Gatos e Cachorros

Este notebook demonstra a aplicação de Transfer Learning usando TensorFlow/Keras para classificar imagens de gatos e cachorros.

**Dataset:** TensorFlow Datasets - cats_vs_dogs

**Modelo Base:** MobileNetV2 (pré-treinado no ImageNet)

**Objetivos:**
- Carregar e preparar o dataset cats_vs_dogs
- Aplicar transfer learning usando MobileNetV2
- Treinar apenas as camadas superiores
- Avaliar o desempenho do modelo
- Visualizar resultados

## 1. Instalação e Importação de Bibliotecas

In [ ]:
# Instalação de dependências (se necessário)
!pip install -q tensorflow tensorflow-datasets matplotlib

In [ ]:
# Importar bibliotecas necessárias
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU disponível: {tf.config.list_physical_devices('GPU')}")

## 2. Carregamento e Preparação dos Dados

In [ ]:
# Configurações
IMG_SIZE = 160  # Tamanho de entrada para MobileNetV2
BATCH_SIZE = 32
SHUFFLE_BUFFER = 1000

# Carregar o dataset cats_vs_dogs
print("Carregando dataset cats_vs_dogs...")
(train_dataset, validation_dataset), info = tfds.load(
    'cats_vs_dogs',
    split=['train[:80%]', 'train[80%:]'],
    with_info=True,
    as_supervised=True,
)

print(f"\nInformações do Dataset:")
print(f"Total de exemplos: {info.splits['train'].num_examples}")
print(f"Exemplos de treino: {tf.data.experimental.cardinality(train_dataset).numpy()}")
print(f"Exemplos de validação: {tf.data.experimental.cardinality(validation_dataset).numpy()}")
print(f"Classes: {info.features['label'].names}")

In [ ]:
# Função de pré-processamento
def preprocess_image(image, label):
    """Redimensiona e normaliza as imagens."""
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = preprocess_input(image)  # Normalização do MobileNetV2
    return image, label

# Aplicar pré-processamento e preparar datasets
train_dataset = train_dataset.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.shuffle(SHUFFLE_BUFFER).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

validation_dataset = validation_dataset.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
validation_dataset = validation_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("Datasets preparados com sucesso!")

## 3. Visualização de Exemplos do Dataset

In [ ]:
# Visualizar algumas imagens do dataset
class_names = info.features['label'].names

plt.figure(figsize=(12, 8))
for images, labels in train_dataset.take(1):
    for i in range(min(9, len(images))):
        plt.subplot(3, 3, i + 1)
        # Reverter a normalização para visualização
        img = images[i].numpy()
        img = (img - img.min()) / (img.max() - img.min())
        plt.imshow(img)
        plt.title(class_names[labels[i]])
        plt.axis('off')
plt.tight_layout()
plt.show()

## 4. Construção do Modelo com Transfer Learning

In [ ]:
# Carregar modelo base MobileNetV2 pré-treinado
print("Carregando MobileNetV2 pré-treinado...")
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,  # Remover camadas de classificação
    weights='imagenet'  # Usar pesos do ImageNet
)

# Congelar as camadas do modelo base
base_model.trainable = False

print(f"\nModelo base carregado com {len(base_model.layers)} camadas")
print(f"Parâmetros treináveis: {base_model.trainable}")

In [ ]:
# Construir modelo completo
print("Construindo modelo completo...")

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')  # Classificação binária
])

# Compilar modelo
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("\nResumo do Modelo:")
model.summary()

## 5. Treinamento do Modelo

In [ ]:
# Configurar callbacks
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=0.00001
)

# Treinar modelo
print("Iniciando treinamento...\n")
EPOCHS = 10

history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS,
    callbacks=[early_stopping, reduce_lr]
)

print("\nTreinamento concluído!")

## 6. Avaliação e Visualização dos Resultados

In [ ]:
# Plotar curvas de treinamento
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Acurácia
axes[0].plot(history.history['accuracy'], label='Treino')
axes[0].plot(history.history['val_accuracy'], label='Validação')
axes[0].set_title('Acurácia do Modelo')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Acurácia')
axes[0].legend()
axes[0].grid(True)

# Loss
axes[1].plot(history.history['loss'], label='Treino')
axes[1].plot(history.history['val_loss'], label='Validação')
axes[1].set_title('Loss do Modelo')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

# Exibir métricas finais
final_train_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]
print(f"\nAcurácia Final:")
print(f"Treino: {final_train_acc:.4f}")
print(f"Validação: {final_val_acc:.4f}")

In [ ]:
# Fazer predições em exemplos de validação
plt.figure(figsize=(15, 10))

for images, labels in validation_dataset.take(1):
    predictions = model.predict(images)
    
    for i in range(min(12, len(images))):
        plt.subplot(3, 4, i + 1)
        
        # Reverter normalização para visualização
        img = images[i].numpy()
        img = (img - img.min()) / (img.max() - img.min())
        plt.imshow(img)
        
        # Obter predição e rótulo real
        predicted_class = 1 if predictions[i] > 0.5 else 0
        true_class = labels[i].numpy()
        confidence = predictions[i][0] if predicted_class == 1 else 1 - predictions[i][0]
        
        # Cor baseada em acerto/erro
        color = 'green' if predicted_class == true_class else 'red'
        
        plt.title(
            f"Real: {class_names[true_class]}\n"
            f"Pred: {class_names[predicted_class]} ({confidence:.2%})",
            color=color
        )
        plt.axis('off')

plt.tight_layout()
plt.show()

## 7. Fine-Tuning (Opcional)

Podemos melhorar ainda mais o modelo descongelando algumas camadas superiores do modelo base e re-treinando com uma taxa de aprendizado menor.

In [ ]:
# Descongelar as últimas camadas do modelo base
print("Aplicando Fine-Tuning...\n")

base_model.trainable = True

# Congelar todas as camadas exceto as últimas 20
for layer in base_model.layers[:-20]:
    layer.trainable = False

# Re-compilar com taxa de aprendizado menor
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print(f"Camadas treináveis: {sum(1 for layer in model.layers if layer.trainable)}")
print(f"Total de parâmetros treináveis: {sum(tf.size(var).numpy() for var in model.trainable_variables)}")

In [ ]:
# Treinar com fine-tuning
FINE_TUNE_EPOCHS = 5

history_fine = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=FINE_TUNE_EPOCHS,
    callbacks=[early_stopping, reduce_lr]
)

print("\nFine-tuning concluído!")

# Exibir métricas finais após fine-tuning
final_train_acc = history_fine.history['accuracy'][-1]
final_val_acc = history_fine.history['val_accuracy'][-1]
print(f"\nAcurácia Final após Fine-Tuning:")
print(f"Treino: {final_train_acc:.4f}")
print(f"Validação: {final_val_acc:.4f}")

## 8. Salvar o Modelo

In [ ]:
# Salvar o modelo treinado
model.save('cats_vs_dogs_model.h5')
print("Modelo salvo com sucesso como 'cats_vs_dogs_model.h5'")

# Também podemos salvar no formato SavedModel
model.save('cats_vs_dogs_model')
print("Modelo salvo no formato SavedModel em 'cats_vs_dogs_model/'")

## 9. Conclusão

Este notebook demonstrou com sucesso a aplicação de Transfer Learning para classificação de imagens de gatos e cachorros usando:

1. **Dataset**: TensorFlow Datasets - cats_vs_dogs
2. **Modelo Base**: MobileNetV2 pré-treinado no ImageNet
3. **Técnica**: Transfer Learning com congelamento inicial seguido de Fine-Tuning opcional

### Vantagens do Transfer Learning:
- ✅ Treinamento mais rápido
- ✅ Menor necessidade de dados
- ✅ Melhor performance inicial
- ✅ Aproveita conhecimento pré-existente

### Próximos Passos:
- Experimentar com outros modelos base (ResNet, EfficientNet, etc.)
- Aplicar data augmentation para melhorar generalização
- Testar diferentes arquiteturas de camadas superiores
- Deploy do modelo em produção

## Referências

- [TensorFlow Transfer Learning Tutorial](https://www.tensorflow.org/tutorials/images/transfer_learning)
- [TensorFlow Datasets - cats_vs_dogs](https://www.tensorflow.org/datasets/catalog/cats_vs_dogs)
- [MobileNetV2 Paper](https://arxiv.org/abs/1801.04381)
- [Keras Applications](https://keras.io/api/applications/)